[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE
#
# GQA 核心思想:
#   MHA: 每个 Q head 都有独立的 K/V head  → num_kv_heads == num_heads
#   MQA: 所有 Q head 共享同一个 K/V head  → num_kv_heads == 1
#   GQA: 每 G 个 Q head 共享一个 K/V head → num_kv_heads = num_heads / G
#
# 好处: KV cache 大小从 num_heads * d_k 降到 num_kv_heads * d_k,
#       推理时显存开销大幅减少, 同时质量接近 MHA.

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.num_heads = num_heads          # Q head 数量 (e.g. 8)
        self.num_kv_heads = num_kv_heads    # KV head 数量 (e.g. 2), 必须能整除 num_heads
        self.d_k = d_model // num_heads     # 每个 head 的维度 (e.g. 32//8 = 4)

        # Q 投影: 输出维度 = d_model = num_heads * d_k (与 MHA 完全一样)
        self.W_q = nn.Linear(d_model, d_model)

        # K/V 投影: 输出维度 = num_kv_heads * d_k (比 MHA 少!)
        # 这就是 GQA 节省参数和显存的关键: K/V 的 head 数更少
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)

        # 输出投影: 与 MHA 一样, 把拼接后的多头结果映射回 d_model
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape  # B=batch, S=seq_len

        # ---- Step 1: 线性投影, 然后 reshape 成多头形式 ----
        # (B, S, d_model) → Linear → (B, S, num_heads * d_k)
        #                           → view  → (B, S, num_heads, d_k)
        #                           → transpose → (B, num_heads, S, d_k)
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        # K/V 只有 num_kv_heads 个 head (比 Q 少)
        # (B, S, num_kv_heads * d_k) → (B, num_kv_heads, S, d_k)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)

        # ---- Step 2: 扩展 KV heads 使其数量与 Q heads 匹配 ----
        # 例如 num_heads=8, num_kv_heads=2 → repeats=4
        # K 从 (B, 2, S, d_k) → repeat_interleave → (B, 8, S, d_k)
        # 即: [kv_head_0, kv_head_1] → [kv0, kv0, kv0, kv0, kv1, kv1, kv1, kv1]
        # 每组 4 个 Q head 共享同一个 KV head
        repeats = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(repeats, dim=1)  # dim=1 是 head 维度
        v = v.repeat_interleave(repeats, dim=1)

        # ---- Step 3: Scaled Dot-Product Attention ----
        # scores = Q @ K^T / sqrt(d_k)
        # 除以 sqrt(d_k) 防止点积值过大导致 softmax 梯度消失
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = torch.softmax(scores, dim=-1)  # (B, num_heads, S, S)
        attn = torch.matmul(weights, v)           # (B, num_heads, S, d_k)

        # ---- Step 4: 合并多头, 输出投影 ----
        # (B, num_heads, S, d_k) → transpose → (B, S, num_heads, d_k)
        #                        → contiguous + view → (B, S, d_model)
        # contiguous() 是因为 transpose 后内存不连续, view 要求连续内存
        out = attn.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)


In [4]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [5]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (5.4ms)
  ✅ [2/5] nn.Linear with correct shapes (1.9ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.2ms)
  ✅ [4/5] KV heads are shared correctly (40.4ms)
  ✅ [5/5] Gradient flow (51.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (100.6ms total)
  Progress saved. Run status() to see your dashboard.

